In [1]:
from srm import catalog

import duckdb
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from srm.bcsd_config import BCSDConfig, PipelineOptions
from srm.orchestration import BCSDOrchestrator
from srm.pipeline import BCSDPipeline

In [2]:
# Define South Africa bounds
df = duckdb.sql(
    """
    install httpfs; load httpfs;
    install spatial; load spatial;
    SELECT name, ST_AsText(geom) as geometry
    FROM ST_Read('https://carbonplan-data.s3.us-west-2.amazonaws.com/countries-50m.json')
    WHERE NAME = 'South Africa'
    """
).df()
df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"])
south_africa_geom = gpd.GeoDataFrame(df, geometry="geometry")

# Set bounds with 2-degree buffer
lon_min, lat_min, lon_max, lat_max = south_africa_geom.total_bounds
bounds = [lat_min - 2, lat_max + 2, lon_min - 2, lon_max + 2]
bounds = [round(entry, 2) for entry in bounds]

print(f"South Africa subset bounds: {bounds}")

South Africa subset bounds: [np.float64(-48.96), np.float64(-20.15), np.float64(14.45), np.float64(39.89)]


In [3]:
# Operational settings (storage paths, environment, runtime flags).
# These live in PipelineOptions and do not affect the computation or cache key.
options = PipelineOptions(
    scratch_dir="s3://carbonplan-scratch/srm/bcsd-cache",
    output_dir="s3://carbonplan-scratch/srm/output",
    version="v6.0.1run20",
    environment="qa",
    verbose=True,
)

In [7]:
# Run-identity configuration (affects computation and is hashed for cache invalidation).
config_dtr = BCSDConfig(
    gcm="CESM2-WACCM",
    variable="dtr",
    ensemble_member="007",  # NCAR-format label used in the SSP245 store
    scenario="SSP245",
    train_period_start=1978,
    train_period_end=2014,
    predict_period_start=2015,
    predict_period_end=2100,
    subset_bounds=tuple(bounds),
)

print(f"Run ID:      {config_dtr.run_id}")
print(f"Config hash: {config_dtr.config_hash}")
print(f"Detrend:     {config_dtr.detrend_data}")

Run ID:      CESM2-WACCM_dtr_007_SSP245_subset
Config hash: ee9a4e035045
Detrend:     True


In [ ]:
# Initialize pipeline
pipeline = BCSDPipeline(config_dtr, options)

# Run all three stages
print("Stage 1: Prepare observations (regrid ERA5 to GCM grid)")
obs_path = pipeline.prepare_observations(force=False)
print(f"✓ Observations cached at: {obs_path}\n")

print("Stage 2: Fit historical (debias and downscale historical period)")
hist_path = pipeline.fit_historical(force=False)
print(f"✓ Historical cached at: {hist_path}\n")

print("Stage 3: Transform scenario (debias and downscale future)")
scenario_path = pipeline.transform_scenario(force=False)
print(f"✓ Scenario output at: {scenario_path}")

Stage 1: Prepare observations (regrid ERA5 to GCM grid)


In [ ]:
# Run-identity configuration (affects computation and is hashed for cache invalidation).
config_tasmax = BCSDConfig(
    gcm="CESM2-WACCM",
    variable="tasmax",
    ensemble_member="007",  # NCAR-format label used in the SSP245 store
    scenario="SSP245",
    train_period_start=1978,
    train_period_end=2014,
    predict_period_start=2015,
    predict_period_end=2100,
    subset_bounds=tuple(bounds),
)

print(f"Run ID:      {config_tasmax.run_id}")
print(f"Config hash: {config_tasmax.config_hash}")
print(f"Detrend:     {config_tasmax.detrend_data}")

In [ ]:
# Initialize pipeline
pipeline = BCSDPipeline(config_tasmax, options)

# Run all three stages
print("Stage 1: Prepare observations (regrid ERA5 to GCM grid)")
obs_path_tasmax = pipeline.prepare_observations(force=False)
print(f"✓ Observations cached at: {obs_path}\n")

print("Stage 2: Fit historical (debias and downscale historical period)")
hist_path_tasmax = pipeline.fit_historical(force=False)
print(f"✓ Historical cached at: {hist_path}\n")

print("Stage 3: Transform scenario (debias and downscale future)")
scenario_path_tasmax = pipeline.transform_scenario(force=False)
print(f"✓ Scenario output at: {scenario_path}")

In [ ]:
from srm.analysis import BCSDRun, load_cached_data

In [ ]:
load_cached_data(scenario_path)